In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gc

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [2]:
BASE_PATH = Path.cwd().parents[0]
GOLD_PATH = BASE_PATH / "data" / "gold"
df = pd.read_csv(GOLD_PATH / "la_port_ml_ready_dataset.csv")

print(df.shape)
df.head()

(233520, 14)


,BerthTime,AvgSOG,VesselDraft,ArrivalHour,ArrivalDayOfWeek,TerminalID,BerthID,DailyCapacityTEU,DraftMargin,LOAMargin,DelayMinutes,IsDelayed,BerthFeasible,CongestionLevel
0,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
1,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
2,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
3,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High
4,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High


In [3]:
df.columns

Index(['BerthTime', 'AvgSOG', 'VesselDraft', 'ArrivalHour', 'ArrivalDayOfWeek',
       'TerminalID', 'BerthID', 'DailyCapacityTEU', 'DraftMargin', 'LOAMargin',
       'DelayMinutes', 'IsDelayed', 'BerthFeasible', 'CongestionLevel'],
      dtype='object')

In [4]:
TARGET = "CongestionLevel"

FEATURES = [
    "BerthTime",
    "ArrivalHour",
    "BerthFeasible",
    "DailyCapacityTEU",
    "ArrivalDayOfWeek"
]

X = df[FEATURES].copy()
y = df[TARGET]

# Encode target if categorical
if y.dtype == "object":
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
X.head()

,BerthTime,ArrivalHour,BerthFeasible,DailyCapacityTEU,ArrivalDayOfWeek
0,232.71,13,1,15298,6
1,232.71,13,1,15298,6
2,232.71,13,1,15298,6
3,232.71,13,1,18334,6
4,232.71,13,1,18334,6


## Logistic Regression

In [6]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)

print("Logistic Regression Accuracy:", lr_acc)


Logistic Regression Accuracy: 0.6960431654676259


## Decision Tree

In [7]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_preds = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)

print("Decision Tree Accuracy:", dt_acc)


Decision Tree Accuracy: 1.0


## Random Forest

In [8]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print("Random Forest Accuracy:", rf_acc)


Random Forest Accuracy: 1.0


## LightGBM

In [9]:
lgb = LGBMClassifier(random_state=42)
lgb.fit(X_train, y_train)

lgb_preds = lgb.predict(X_test)
lgb_acc = accuracy_score(y_test, lgb_preds)

print("LightGBM Accuracy:", lgb_acc)


[LightGBM] [Info] Number of positive: 56784, number of negative: 130032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 543
[LightGBM] [Info] Number of data points in the train set: 186816, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303957 -> initscore=-0.828526
[LightGBM] [Info] Start training from score -0.828526
LightGBM Accuracy: 0.9741778006166495


## XGBOOST

In [10]:
xgb = XGBClassifier(
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)

print("XGBoost Accuracy:", xgb_acc)


XGBoost Accuracy: 0.9944972593353888


## Comparing the Initial Models

In [11]:
initial_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "Accuracy": [
        lr_acc,
        dt_acc,
        rf_acc,
        xgb_acc,
        lgb_acc
    ]
})

initial_results.sort_values("Accuracy", ascending=False)


,Model,Accuracy
1,Decision Tree,1.000000
2,Random Forest,1.000000
3,XGBoost,0.994497
4,LightGBM,0.974178
0,Logistic Regression,0.696043


## Tuned – RANDOM FOREST

In [12]:
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=9,
    min_samples_split=70,
    min_samples_leaf=30,
    max_features=0.4,
    bootstrap=True,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)
rf_tuned_acc = accuracy_score(y_test, rf_tuned.predict(X_test))

print("Tuned RF Accuracy:", rf_tuned_acc)

Tuned RF Accuracy: 0.8346180198698184


In [13]:
print("Tuned RF Accuracy:", rf_tuned_acc)

# Classification Report
print("\nClassification Report (Tuned Random Forest):")
print(classification_report(y_test, rf_tuned.predict(X_test)))

Tuned RF Accuracy: 0.8346180198698184

Classification Report (Tuned Random Forest):
              precision    recall  f1-score   support

           0       0.95      0.80      0.87     32508
           1       0.67      0.91      0.77     14196

    accuracy                           0.83     46704
   macro avg       0.81      0.86      0.82     46704
weighted avg       0.87      0.83      0.84     46704



## Tuned – LIGHTGBM

lgb_params = {
    "num_leaves": [31, 50],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
    "max_depth": [-1, 10]
}

lgb_gs = GridSearchCV(
    LGBMClassifier(random_state=42),
    lgb_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

lgb_gs.fit(X_train, y_train)

best_lgb = lgb_gs.best_estimator_
lgb_gs_acc = accuracy_score(y_test, best_lgb.predict(X_test))

print("Best LGB Params:", lgb_gs.best_params_)
print("Tuned LGB Accuracy:", lgb_gs_acc)


In [14]:
lgb_tuned = LGBMClassifier(
    num_leaves=50,
    learning_rate=0.05,
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
lgb_tuned.fit(X_train, y_train)


lgb_tuned_acc = accuracy_score(y_test, lgb_tuned.predict(X_test))


print("Tuned LGB Accuracy:", lgb_tuned_acc)

[LightGBM] [Info] Number of positive: 56784, number of negative: 130032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001240 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 543
[LightGBM] [Info] Number of data points in the train set: 186816, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Tuned LGB Accuracy: 0.9748843782117164


## Tuned – XGBOOST

xgb_params = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
    "subsample": [0.8, 1.0]
}

xgb_gs = GridSearchCV(
    XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y)),
        eval_metric="mlogloss",
        random_state=42
    ),
    xgb_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

xgb_gs.fit(X_train, y_train)

best_xgb = xgb_gs.best_estimator_
xgb_gs_acc = accuracy_score(y_test, best_xgb.predict(X_test))

print("Best XGB Params:", xgb_gs.best_params_)
print("Tuned XGB Accuracy:", xgb_gs_acc)


In [18]:
xgb_tuned = XGBClassifier(
    objective="binary:logistic",
    max_depth=8,
    learning_rate=0.02,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_tuned.fit(X_train, y_train)

y_pred = xgb_tuned.predict(X_test)
xgb_tuned_acc = accuracy_score(y_test, y_pred)

print("Tuned XGB Accuracy:", xgb_tuned_acc)

Tuned XGB Accuracy: 0.937071771154505


rf_realistic = RandomForestClassifier(
    n_estimators=200,
    max_depth=9,
    min_samples_split=50,
    min_samples_leaf=30,
    max_features=0.4,
    bootstrap=True,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_realistic.fit(X_train, y_train)

y_pred_real = rf_realistic.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_real))
print("\nClassification Report:\n", classification_report(y_test, y_pred_real))

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 7, 9, 11],
    "min_samples_split": [50, 70, 100],
    "min_samples_leaf": [20, 30, 50],
    "max_features": [0.4],          # fixed value
    "bootstrap": [True],             # fixed value
    "class_weight": ["balanced"]     # fixed value
}


rf_gs = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


rf_gs.fit(X_train, y_train)

best_rf = rf_gs.best_estimator_
rf_gs_acc = accuracy_score(y_test, best_rf.predict(X_test))

print("Best RF Params:", rf_gs.best_params_)
print("Tuned RF Accuracy:", rf_gs_acc)


In [19]:
tuned_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "Accuracy": [
        rf_tuned_acc,
        xgb_tuned_acc,
        lgb_tuned_acc
    ]
})

tuned_results.sort_values("Accuracy", ascending=False)


,Model,Accuracy
2,LightGBM,0.974884
1,XGBoost,0.937072
0,Random Forest,0.834618


In [20]:
import joblib
MODEL_PATH = BASE_PATH / "models"

joblib.dump(xgb_tuned, MODEL_PATH / "xgb_congestion.pkl")

['E:\\C-DAC\\Major Project\\AI-Based-Maritime-Port-Intelligence\\models\\xgb_congestion.pkl']

In [21]:
# Load trained GridSearch model
model = joblib.load(MODEL_PATH / "xgb_congestion.pkl")

In [22]:
# Example: new vessel arrival data
new_data = {
    "BerthTime": 180.5,
    "ArrivalHour": 14,
    "BerthFeasible": 1,
    "DailyCapacityTEU": 16000,
    "ArrivalDayOfWeek": 2
}

# Convert to DataFrame
X_new = pd.DataFrame([new_data], columns=FEATURES)

# Predict
pred = model.predict(X_new)
pred_proba = model.predict_proba(X_new)

print("Predicted class (encoded):", pred[0])
print("Prediction probabilities:", pred_proba)


Predicted class (encoded): 1
Prediction probabilities: [[0.2946527 0.7053473]]


In [23]:


# ------------------ FEATURES (same order as training) ------------------
FEATURES = [
    "BerthTime",
    "ArrivalHour",
    "BerthFeasible",
    "DailyCapacityTEU",
    "ArrivalDayOfWeek",
]

# ------------------ SCENARIOS ------------------
scenarios = [
    [15, 8, 1, 9000, 1],
    [20, 10, 1, 8500, 2],
    [25, 12, 1, 8000, 3],
    [30, 14, 1, 7800, 4],
    [35, 16, 1, 7600, 5],
    [40, 18, 0, 7400, 5],
    [45, 20, 0, 7200, 6],
    [50, 22, 0, 7000, 6],
    [28, 1, 1, 8800, 0],
    [18, 5, 1, 9200, 0],

    [22, 7, 1, 8900, 1],
    [26, 9, 1, 8700, 2],
    [30, 11, 1, 8400, 3],
    [34, 13, 1, 8200, 4],
    [38, 15, 0, 7900, 5],
    [42, 17, 0, 7600, 5],
    [46, 19, 0, 7400, 6],
    [50, 21, 0, 7100, 6],
    [24, 23, 1, 8600, 0],
    [16, 3, 1, 9300, 0],

    [14, 6, 1, 9500, 1],
    [19, 8, 1, 9000, 2],
    [23, 10, 1, 8700, 3],
    [27, 12, 1, 8500, 4],
    [31, 14, 1, 8200, 5],
    [36, 16, 0, 8000, 5],
    [41, 18, 0, 7700, 6],
    [47, 20, 0, 7400, 6],
    [29, 22, 1, 8800, 0],
    [17, 2, 1, 9400, 0],

    [21, 9, 1, 9100, 1],
    [25, 11, 1, 8800, 2],
    [29, 13, 1, 8500, 3],
    [33, 15, 1, 8200, 4],
    [37, 17, 0, 7900, 5],
    [41, 19, 0, 7600, 5],
    [45, 21, 0, 7300, 6],
    [50, 23, 0, 7000, 6],
    [26, 4, 1, 9000, 0],
    [18, 6, 1, 9300, 0],

    [12, 7, 1, 9600, 1],
    [17, 9, 1, 9200, 2],
    [22, 11, 1, 8900, 3],
    [27, 13, 1, 8600, 4],
    [32, 15, 1, 8300, 5],
    [37, 17, 0, 8000, 5],
    [42, 19, 0, 7700, 6],
    [48, 21, 0, 7400, 6],
    [30, 23, 1, 8800, 0],
    [16, 1, 1, 9500, 0],

    [20, 8, 1, 9100, 1],
    [24, 10, 1, 8800, 2],
    [28, 12, 1, 8500, 3],
    [32, 14, 1, 8200, 4],
    [36, 16, 0, 7900, 5],
    [40, 18, 0, 7600, 5],
    [45, 20, 0, 7300, 6],
    [50, 22, 0, 7000, 6],
    [27, 0, 1, 9000, 0],
    [19, 4, 1, 9400, 0],

    [13, 6, 1, 9700, 1],
    [18, 8, 1, 9300, 2],
    [23, 10, 1, 9000, 3],
    [28, 12, 1, 8700, 4],
    [33, 14, 1, 8400, 5],
    [38, 16, 0, 8100, 5],
    [43, 18, 0, 7800, 6],
    [49, 20, 0, 7500, 6],
    [31, 22, 1, 8900, 0],
    [17, 2, 1, 9600, 0]
]



# ------------------ DATAFRAME ------------------
X_batch = pd.DataFrame(scenarios, columns=FEATURES)

# ------------------ PREDICTION ------------------
preds = model.predict(X_batch)
probas = model.predict_proba(X_batch)

# ------------------ LABEL DECODING ------------------
label_map = {
    0: "Low",
    1: "High"
}

# ------------------ OUTPUT ------------------
for i, (pred, prob) in enumerate(zip(preds, probas)):
    print(f"Scenario {i+1}")
    print(f"  Predicted Congestion Level : {label_map[pred]}")
    print(f"  Confidence (Low/Med/High) : {prob.round(2)}")
    print("-" * 45)


Scenario 1
  Predicted Congestion Level : High
  Confidence (Low/Med/High) : [0.39 0.61]
---------------------------------------------
Scenario 2
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.64 0.36]
---------------------------------------------
Scenario 3
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.76 0.24]
---------------------------------------------
Scenario 4
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.86 0.14]
---------------------------------------------
Scenario 5
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.93 0.07]
---------------------------------------------
Scenario 6
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.76 0.24]
---------------------------------------------
Scenario 7
  Predicted Congestion Level : Low
  Confidence (Low/Med/High) : [0.91 0.09]
---------------------------------------------
Scenario 8
  Predicted Congestion Level : Low
  Confidence (L

In [24]:
joblib.dump(rf_tuned, MODEL_PATH / "rf_congestion.pkl")
model_rf = joblib.load(MODEL_PATH / "rf_congestion.pkl")

In [25]:


# ------------------ FEATURES (same order as training) ------------------
FEATURES = [
    "BerthTime",
    "ArrivalHour",
    "BerthFeasible",
    "DailyCapacityTEU",
    "ArrivalDayOfWeek",
]

# ------------------ SCENARIOS ------------------
scenarios = [
    [15, 8, 1, 9000, 1],
    [20, 10, 1, 8500, 2],
    [25, 12, 1, 8000, 3],
    [30, 14, 1, 7800, 4],
    [35, 16, 1, 7600, 5],
    [40, 18, 0, 7400, 5],
    [45, 20, 0, 7200, 6],
    [50, 22, 0, 7000, 6],
    [28, 1, 1, 8800, 0],
    [18, 5, 1, 9200, 0],

    [22, 7, 1, 8900, 1],
    [26, 9, 1, 8700, 2],
    [30, 11, 1, 8400, 3],
    [34, 13, 1, 8200, 4],
    [38, 15, 0, 7900, 5],
    [42, 17, 0, 7600, 5],
    [46, 19, 0, 7400, 6],
    [50, 21, 0, 7100, 6],
    [24, 23, 1, 8600, 0],
    [16, 3, 1, 9300, 0],

    [14, 6, 1, 9500, 1],
    [19, 8, 1, 9000, 2],
    [23, 10, 1, 8700, 3],
    [27, 12, 1, 8500, 4],
    [31, 14, 1, 8200, 5],
    [36, 16, 0, 8000, 5],
    [41, 18, 0, 7700, 6],
    [47, 20, 0, 7400, 6],
    [29, 22, 1, 8800, 0],
    [17, 2, 1, 9400, 0],

    [21, 9, 1, 9100, 1],
    [25, 11, 1, 8800, 2],
    [29, 13, 1, 8500, 3],
    [33, 15, 1, 8200, 4],
    [37, 17, 0, 7900, 5],
    [41, 19, 0, 7600, 5],
    [45, 21, 0, 7300, 6],
    [50, 23, 0, 7000, 6],
    [26, 4, 1, 9000, 0],
    [18, 6, 1, 9300, 0],

    [12, 7, 1, 9600, 1],
    [17, 9, 1, 9200, 2],
    [22, 11, 1, 8900, 3],
    [27, 13, 1, 8600, 4],
    [32, 15, 1, 8300, 5],
    [37, 17, 0, 8000, 5],
    [42, 19, 0, 7700, 6],
    [48, 21, 0, 7400, 6],
    [30, 23, 1, 8800, 0],
    [16, 1, 1, 9500, 0],

    [20, 8, 1, 9100, 1],
    [24, 10, 1, 8800, 2],
    [28, 12, 1, 8500, 3],
    [32, 14, 1, 8200, 4],
    [36, 16, 0, 7900, 5],
    [40, 18, 0, 7600, 5],
    [45, 20, 0, 7300, 6],
    [50, 22, 0, 7000, 6],
    [27, 0, 1, 9000, 0],
    [19, 4, 1, 9400, 0],

    [13, 6, 1, 9700, 1],
    [18, 8, 1, 9300, 2],
    [23, 10, 1, 9000, 3],
    [28, 12, 1, 8700, 4],
    [33, 14, 1, 8400, 5],
    [38, 16, 0, 8100, 5],
    [43, 18, 0, 7800, 6],
    [49, 20, 0, 7500, 6],
    [31, 22, 1, 8900, 0],
    [17, 2, 1, 9600, 0]
]



# ------------------ DATAFRAME ------------------
X_batch = pd.DataFrame(scenarios, columns=FEATURES)

# ------------------ PREDICTION ------------------
preds = model_rf.predict(X_batch)
probas = model_rf.predict_proba(X_batch)

# ------------------ LABEL DECODING ------------------
label_map = {
    0: "Low",
    1: "High"
}

# ------------------ OUTPUT ------------------
for i, (pred, prob) in enumerate(zip(preds, probas)):
    print(f"Scenario {i+1}")
    print(f"  Predicted Congestion Level : {label_map[pred]}")
    print(f"  Confidence (Low/High) : {prob.round(2)}")
    print("-" * 45)


Scenario 1
  Predicted Congestion Level : High
  Confidence (Low/High) : [0.39 0.61]
---------------------------------------------
Scenario 2
  Predicted Congestion Level : Low
  Confidence (Low/High) : [0.51 0.49]
---------------------------------------------
Scenario 3
  Predicted Congestion Level : High
  Confidence (Low/High) : [0.45 0.55]
---------------------------------------------
Scenario 4
  Predicted Congestion Level : Low
  Confidence (Low/High) : [0.52 0.48]
---------------------------------------------
Scenario 5
  Predicted Congestion Level : Low
  Confidence (Low/High) : [0.65 0.35]
---------------------------------------------
Scenario 6
  Predicted Congestion Level : Low
  Confidence (Low/High) : [0.51 0.49]
---------------------------------------------
Scenario 7
  Predicted Congestion Level : Low
  Confidence (Low/High) : [0.57 0.43]
---------------------------------------------
Scenario 8
  Predicted Congestion Level : High
  Confidence (Low/High) : [0.48 0.52]
---